# Geospatial Data Quality & Governance Framework — QC Workflow
Rule-based quality assessment of OpenStreetMap buildings, roads and POIs for the Dammam–Khobar urban area (Saudi Arabia).

**Input:** [Geofabrik GCC States](https://download.geofabrik.de/asia/gcc-states.html) extract, Shapefile format (16-09-2026).
**Output:** `GDQ_Rerun.gdb` with the study area, raw layers, `QC_Results` and `QC_Violations`, plus `QC_Results_rerun.csv`.
**Requires:** ArcGIS Pro 3.x (ArcPy).

**How to run:** set `SRC` and `PRJ` in the first code cell, then *Cell → Run All*. The last check cell compares every rule with the published results (15 of 15 matched on the 16-09-2026 extract).

Data © OpenStreetMap contributors, ODbL. Author: Norah Alhumud.

## 1. Setup

In [ ]:
import arcpy, os
from collections import Counter

SRC = r"C:\path\to\gcc-states-260916-free.shp"   # <-- folder of the Geofabrik GCC States shapefiles
PRJ = r"C:\path\to\output_folder"                  # <-- any folder; GDQ_Rerun.gdb is created here
GDB = os.path.join(PRJ, "GDQ_Rerun.gdb")
if not arcpy.Exists(GDB):
    arcpy.management.CreateFileGDB(PRJ, "GDQ_Rerun.gdb")

arcpy.env.workspace = GDB
arcpy.env.overwriteOutput = True
UTM = arcpy.SpatialReference(32639)      # WGS 84 / UTM zone 39N
WGS = arcpy.SpatialReference(4326)
sc = arcpy.env.scratchGDB
S = lambda n: os.path.join(SRC, n)       # source shapefile
G = lambda n: os.path.join(GDB, n)       # output in GDQ_Rerun.gdb
print("Output:", GDB)

## 2. Admin boundaries (Dammam + Khobar governorates)
OSM relations verified by area and location: Dammam 12399708 (1,846.8 km²), Khobar 12400302 (197.7 km²).

In [ ]:
ADMIN = G("Admin_Dammam_Khobar")
arcpy.conversion.ExportFeatures(S("gis_osm_adminareas_a_free_1.shp"), ADMIN,
                                "osm_id IN ('12399708', '12400302')")
bl = arcpy.management.MakeFeatureLayer(S("gis_osm_buildings_a_free_1.shp"), "rr_bl_admin")
arcpy.management.SelectLayerByLocation(bl, "HAVE_THEIR_CENTER_IN", ADMIN)
print("admin polygons:", arcpy.management.GetCount(ADMIN)[0], "| buildings in admin:", arcpy.management.GetCount(bl)[0])

## 3. Urban study area
Building points (INSIDE) aggregated at 500 m in UTM 39N; polygons of 1 km² or more are kept.

In [ ]:
pts, agg = os.path.join(sc, "rr_pts"), os.path.join(sc, "rr_agg")
with arcpy.EnvManager(outputCoordinateSystem=UTM):
    arcpy.management.FeatureToPoint(bl, pts, "INSIDE")
arcpy.cartography.AggregatePoints(in_features=pts, out_feature_class=agg, aggregation_distance="500 Meters")
arcpy.management.AddField(agg, "km2", "DOUBLE")
arcpy.management.CalculateGeometryAttributes(agg, [["km2", "AREA_GEODESIC"]], area_unit="SQUARE_KILOMETERS")
a = [r[0] for r in arcpy.da.SearchCursor(agg, ["km2"])]
SA = G("Study_Area")
with arcpy.EnvManager(outputCoordinateSystem=WGS):
    arcpy.conversion.ExportFeatures(agg, SA, "km2 >= 1")
print("aggregated:", len(a), "polygons,", round(sum(a), 1), "km2 | study area:",
      sum(x >= 1 for x in a), "polygons,", round(sum(x for x in a if x >= 1), 1), "km2")
if sum(x >= 1 for x in a) > 20:
    raise RuntimeError("Too many study-area polygons: the admin selection was not applied. Stop and check cell 2.")

## 4. Raw layers inside the study area (originals stay untouched)

In [ ]:
B, R, P = G("Buildings_Raw"), G("Roads_Raw"), G("POIs_Raw")
for shp, out in [("gis_osm_buildings_a_free_1.shp", B), ("gis_osm_roads_free_1.shp", R), ("gis_osm_pois_free_1.shp", P)]:
    lyr = arcpy.management.MakeFeatureLayer(S(shp), "rr_" + os.path.basename(out))
    arcpy.management.SelectLayerByLocation(lyr, "HAVE_THEIR_CENTER_IN", SA)
    arcpy.conversion.ExportFeatures(lyr, out)
    print(os.path.basename(out), arcpy.management.GetCount(out)[0])
nb_ = int(arcpy.management.GetCount(B)[0])
if nb_ > 200000:
    raise RuntimeError(f"Buildings_Raw has {nb_:,} features: the study-area selection was not applied. Stop and check cells 2-3.")

## 5. Results tables and logging helper

In [ ]:
qc, qv = G("QC_Results"), G("QC_Violations")
for t in (qc, qv):
    if arcpy.Exists(t): arcpy.management.Delete(t)
arcpy.management.CreateTable(GDB, "QC_Results")
for f, t, l in [("layer", "TEXT", 40), ("rule_id", "TEXT", 10), ("dimension", "TEXT", 20), ("rule", "TEXT", 120),
                ("severity", "TEXT", 10), ("checked", "LONG", None), ("failed", "LONG", None)]:
    arcpy.management.AddField(qc, f, t, field_length=l)
arcpy.management.CreateTable(GDB, "QC_Violations")
arcpy.management.AddField(qv, "rule_id", "TEXT", field_length=10)
arcpy.management.AddField(qv, "osm_id", "TEXT", field_length=20)

RES = {}
def log(layer, rid, dim, rule, sev, checked, bad):
    bad = list(bad)
    RES[rid] = (checked, len(bad), [str(o) for o in bad])
    with arcpy.da.InsertCursor(qc, ["layer", "rule_id", "dimension", "rule", "severity", "checked", "failed"]) as c:
        c.insertRow((layer, rid, dim, rule, sev, checked, len(bad)))
    with arcpy.da.InsertCursor(qv, ["rule_id", "osm_id"]) as c:
        for o in bad: c.insertRow((rid, str(o)))
    print(f"{rid:4} {sev:8} checked {checked:>7,}  failed {len(bad):>5,}")

txt = lambda v: (v or "").strip()     # Geofabrik stores empty text as a single space
print("ready")

## 6. Buildings rules (B1–B4)

In [ ]:
bname = "Dammam_Buildings_Raw"
rows = [(o, g) for o, g in arcpy.da.SearchCursor(B, ["osm_id", "SHAPE@"])]
n = len(rows)
cnt = Counter(o for o, _ in rows)
log(bname, "B1", "Uniqueness", "osm_id unique", "Error", n, [o for o, k in cnt.items() if k > 1])

chk = os.path.join(sc, "rr_chk")
arcpy.management.CheckGeometry(B, chk)
bad_oid = {r[0] for r in arcpy.da.SearchCursor(chk, ["FEATURE_ID"])}
oid2id = {r[0]: r[1] for r in arcpy.da.SearchCursor(B, ["OID@", "osm_id"])}
log(bname, "B2", "Geometry", "valid geometry (CheckGeometry)", "Error", n, [oid2id[i] for i in bad_oid if i in oid2id])

area = [(o, g.getArea("GEODESIC", "SQUAREMETERS") if g else 0) for o, g in rows]
log(bname, "B3", "Plausibility", "area >= 10 m2", "Warning", n, [o for o, a in area if a < 10])
log(bname, "B4", "Plausibility", "area <= 50,000 m2", "Review", n, [o for o, a in area if a > 50000])

## 7. Roads rules (R1–R7)

In [ ]:
rname = "Dammam_Roads_Raw"
F = ["osm_id", "fclass", "name", "ref", "oneway", "maxspeed", "layer", "bridge", "tunnel"]
rd = [dict(zip(F, r)) for r in arcpy.da.SearchCursor(R, F)]
n = len(rd)
MAJOR = ("motorway", "trunk", "primary", "secondary")
named = lambda r: bool(txt(r["name"]) or txt(r["ref"]))

cnt = Counter(r["osm_id"] for r in rd)
log(rname, "R1", "Uniqueness", "osm_id unique", "Error", n, [o for o, k in cnt.items() if k > 1])
maj = [r for r in rd if r["fclass"] in MAJOR]
R2A_BAD = [r["osm_id"] for r in maj if not named(r)]
log(rname, "R2a", "Completeness", "motorway/trunk/primary/secondary have name or ref", "Error", len(maj), R2A_BAD)
ter = [r for r in rd if r["fclass"] == "tertiary"]
log(rname, "R2b", "Completeness", "tertiary has name or ref", "Warning", len(ter), [r["osm_id"] for r in ter if not named(r)])
log(rname, "R3", "Validity", "oneway in F/T/B", "Error", n, [r["osm_id"] for r in rd if txt(r["oneway"]) not in ("F", "T", "B")])
ms = [r for r in rd if (r["maxspeed"] or 0) > 0]
log(rname, "R4", "Validity", "maxspeed 20-140 when set (0 = not set)", "Error", len(ms), [r["osm_id"] for r in ms if not 20 <= r["maxspeed"] <= 140])
tu = [r for r in rd if txt(r["tunnel"]) == "T"]
log(rname, "R5", "Consistency", "tunnel=T implies layer<0", "Warning", len(tu), [r["osm_id"] for r in tu if not (r["layer"] or 0) < 0])
br = [r for r in rd if txt(r["bridge"]) == "T"]
log(rname, "R6", "Consistency", "bridge=T implies layer>0", "Warning", len(br), [r["osm_id"] for r in br if not (r["layer"] or 0) > 0])
ly = [r for r in rd if (r["layer"] or 0) != 0]
log(rname, "R7", "Consistency", "layer>0 implies bridge, layer<0 implies tunnel", "Warning", len(ly),
    [r["osm_id"] for r in ly if (r["layer"] > 0 and txt(r["bridge"]) != "T") or (r["layer"] < 0 and txt(r["tunnel"]) != "T")])

## 8. R8 — are unnamed major roads isolated from named major roads?

In [ ]:
ids = [str(o) for o in R2A_BAD]
q = "osm_id IN (" + ",".join("'" + i + "'" for i in ids) + ")"
named = arcpy.management.MakeFeatureLayer(R, "rr_named_hi", "fclass IN ('motorway','trunk','primary','secondary') AND name <> ' '")
viol = arcpy.management.MakeFeatureLayer(R, "rr_viol_hi", q)
touch = arcpy.management.SelectLayerByLocation(viol, "BOUNDARY_TOUCHES", named)
touch_ids = {str(r[0]) for r in arcpy.da.SearchCursor(touch, ["osm_id"])}
print("touching a named major road:", len(touch_ids), "of", len(ids))
log(rname, "R8", "Completeness", "R2a violations with no named major neighbour (needs source check)", "Review",
    len(ids), [i for i in ids if i not in touch_ids])

## 9. POIs rules (P1–P2)

In [ ]:
pname = "Dammam_POIs_Raw"
pois = [dict(zip(["osm_id", "code", "fclass", "name"], r)) for r in arcpy.da.SearchCursor(P, ["osm_id", "code", "fclass", "name"])]
cnt = Counter(p["osm_id"] for p in pois)
log(pname, "P1", "Uniqueness", "osm_id unique", "Error", len(pois), [o for o, k in cnt.items() if k > 1])

biz = ("supermarket", "restaurant", "fast_food", "cafe", "bakery", "pharmacy", "bank", "hotel",
       "mall", "clothes", "hairdresser", "car_dealership")
com = [p for p in pois if p["fclass"] in biz]
log(pname, "P2", "Completeness", "commercial POIs have name", "Warning", len(com), [p["osm_id"] for p in com if not txt(p["name"])])

## 10. Check against the published results

In [ ]:
PUBLISHED = {"B1": (91212, 0), "B2": (91212, 1), "B3": (91212, 25), "B4": (91212, 3),
             "R1": (18403, 0), "R2a": (686, 252), "R2b": (705, 518), "R3": (18403, 0), "R4": (278, 0),
             "R5": (68, 42), "R6": (99, 0), "R7": (137, 12), "R8": (252, 103),
             "P1": (1203, 0), "P2": (685, 213)}
ok = 0
for rid, (c, f) in PUBLISHED.items():
    got = RES.get(rid, (None, None))[:2]
    match = got == (c, f); ok += match
    print(f"{rid:4} published {c:>6}/{f:<4} rerun {str(got[0]):>6}/{str(got[1]):<4} {'OK' if match else '<< DIFFERENT'}")
print(f"\n{ok} of {len(PUBLISHED)} rules match")


## 11. Export results for the dashboard

In [ ]:
out_csv = os.path.join(PRJ, "QC_Results_rerun.csv")
arcpy.conversion.ExportTable(qc, out_csv)
print("saved:", out_csv)